# Training Tabular Data

In [1]:
# Importing data
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# Importing cleaned dataset
df = pd.read_csv('../data/anime_dataset_clean.csv')

In [3]:
# Independent & Dependent features
X = df.drop(columns=['title','synopsis','high_rated'])
y = df['high_rated']
print(f"Shape of Independent feature:{X.shape}\nShape of dependent feature:{y.shape}")

Shape of Independent feature:(18158, 9)
Shape of dependent feature:(18158,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y,random_state=42,test_size=0.25)

## Data Standardizing

### Categorical and Numerical

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.pipeline import Pipeline

numeric_feature = ['episodes','duration_mins']
categorical_feature = ['type','source','rating']
# Instances
num_transformer = StandardScaler() # list not df
cat_transformer = OneHotEncoder(sparse_output=False)

# Numeric standardizing
preprocessor = ColumnTransformer([
    ("StandardScaler",num_transformer,numeric_feature),
    ("OneHotEncoder",cat_transformer,categorical_feature),
],remainder='drop')
# FIT and TRANSFORM on train
X_train_preprocessed_arr = preprocessor.fit_transform(X_train)
preprocessor_cols = preprocessor.get_feature_names_out()  # new dynamically generated column names
X_train_standard = pd.DataFrame(X_train_preprocessed_arr, columns=preprocessor_cols, index=X_train.index) # wrap new arr back to df, reattaching the column good for merging with mlb

# strictly TRANSFORM on test
X_test_preprocessed_arr = preprocessor.transform(X_test)
X_test_standard = pd.DataFrame(X_test_preprocessed_arr, columns=preprocessor_cols, index=X_test.index)

### Encoding Multi-Select Features

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb_cols = ['studios','genres','themes']
mlb_train_dfs = []
mlb_test_dfs = []
# studio encoding
for col in mlb_cols:
    mlb = MultiLabelBinarizer()
    # transforming train data
    train_encoded = mlb.fit_transform(X_train[col])
    cols = [f'{col}_{c}' for c in mlb.classes_]
    mlb_train_dfs.append(pd.DataFrame(train_encoded,columns=cols,index=X_train.index))
    # transforming test data
    test_encoded = mlb.transform(X_test[col])
    mlb_test_dfs.append(pd.DataFrame(test_encoded,columns=cols,index=X_test.index))
    
X_train_mlb = pd.concat(mlb_train_dfs, axis=1)
X_test_mlb = pd.concat(mlb_test_dfs, axis=1)


In [7]:
### Merging
X_train_final = pd.concat([X_train_standard,X_train_mlb],axis=1)
X_test_final = pd.concat([X_test_standard,X_test_mlb],axis=1)


## Model Training

In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. Define models with balanced class handling
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
}

results = []

# 2. Loop cleanly using .items()
for model_name, model in models.items():
    # Fit model on training data
    model.fit(X_train_final, y_train)
    
    # Generate predictions
    y_train_pred = model.predict(X_train_final)
    y_test_pred = model.predict(X_test_final)
    
    # Generate probability predictions for ROC-AUC
    y_train_proba = model.predict_proba(X_train_final)[:, 1]
    y_test_proba = model.predict_proba(X_test_final)[:, 1]
    
    # Store metrics in dictionary
    results.append({
        "Model": model_name,
        "Train Macro F1": f1_score(y_train, y_train_pred, average='macro'),
        "Test Macro F1": f1_score(y_test, y_test_pred, average='macro'),
        "Train ROC-AUC": roc_auc_score(y_train, y_train_proba),
        "Test ROC-AUC": roc_auc_score(y_test, y_test_proba),
        "Test Precision (Class 1)": precision_score(y_test, y_test_pred),
        "Test Recall (Class 1)": recall_score(y_test, y_test_pred)
    })

# 3. Display as a clean comparison DataFrame
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

              Model  Train Macro F1  Test Macro F1  Train ROC-AUC  Test ROC-AUC  Test Precision (Class 1)  Test Recall (Class 1)
Logistic Regression        0.737630       0.735798       0.849870      0.846796                  0.559788               0.799698
      Random Forest        0.983701       0.787876       0.996412      0.875006                  0.675676               0.736961


#### Observation
- Logistic Regression has high-stability with minimal train-test gap in macro F1 Score (`0.738` vs `0.735` for train and test respectively). But it is capped by linear decision boundary.
- Random Forest Classifier performed better than compare to Logistic Regression in every performance metrics.Test F1 score is `0.7878` compare to `0.7357` of Logistic Regression. Same can be said for *Test ROC-AUC* `0.875` compare to Logistic regression `0.846`.
- Major key issue for baseline Random Forest is it is overfitting (Train F1: `0.983` vs Test F1: `0.787`).

## Hyper Parameter Tuning

### Cross Validation & Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score

cv_strategy =  StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_distribution = {
    'n_estimators': [100, 200, 300],          # Number of trees in the forest
    'max_depth': [10, 15, 20, None],          # Limits tree depth
    'min_samples_split': [5, 10, 20],         # Min samples to create a split
    'min_samples_leaf': [2, 5, 10],           # Min samples required at a leaf
    'max_features': ['sqrt', 0.3, 0.5]        # Subsampling features per split
}

rf_base  = RandomForestClassifier(class_weight='balanced',random_state=42, n_jobs=-1)

rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distribution,
    n_iter=20,                       # Tests 20 random parameter combinations
    scoring='f1_macro',              # Optimizes directly for Macro F1
    cv=cv_strategy,
    random_state=42,
    n_jobs=2,                       # Uses all CPU cores
    verbose=1
)

rf_random_search.fit(X_train_final,y_train)
best_rf = rf_random_search.best_estimator_
print("=== Best Hyperparameters Found ===")
print(rf_random_search.best_params_)
print(f'\nBest 5-fold CV Macro F1: {rf_random_search.best_score_:.4f}')

In [10]:
y_train_pred = best_rf.predict(X_train_final)
y_test_pred = best_rf.predict(X_test_final)

y_train_proba = best_rf.predict_proba(X_train_final)[:, 1]
y_test_proba = best_rf.predict_proba(X_test_final)[:, 1]

print("=== Tuned Random Forest Evaluation ===")
print(f"Train Macro F1: {f1_score(y_train, y_train_pred, average='macro'):.4f}")
print(f"Test Macro F1 : {f1_score(y_test, y_test_pred, average='macro'):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, y_test_proba):.4f}")

print("\n=== Test Classification Report ===")
print(classification_report(y_test, y_test_pred))

=== Tuned Random Forest Evaluation ===
Train Macro F1: 0.9172
Test Macro F1 : 0.7835
Test ROC-AUC  : 0.8819

=== Test Classification Report ===
              precision    recall  f1-score   support

           0       0.90      0.82      0.86      3217
           1       0.65      0.78      0.71      1323

    accuracy                           0.81      4540
   macro avg       0.77      0.80      0.78      4540
weighted avg       0.83      0.81      0.82      4540



#### Observation:
- Tuning reduce the train/test gap `0.192` to `0.134`.
- There is slightly improved performance metrics for tuned model ROC-AUC (`0.875->0.882`) & but barely for Test Macro F1(`0.791 -> 0.784`)


## Conclusion
- For Tabular Baseline Random Forest Classifier (tuned) is selected over Logistic Regression because of better Test Macro F1 adn ROC-AUC. Despite a residual train/test gap decrease.

In [13]:
import joblib
import os
os.makedirs('models', exist_ok=True)
# Save the trained model and preprocessors
joblib.dump(best_rf, '../models/tabular_baseline_model.pkl')
joblib.dump(preprocessor, '../models/tabular_baseline_preprocessor.pkl')

['../models/tabular_baseline_preprocessor.pkl']